In [10]:
!pip install -q openai

In [11]:
from google.colab import userdata

api_key = userdata.get("KODEKLOUD_API_KEY")

print("API key loaded:", bool(api_key))

API key loaded: True


There are two things here:
- TypeSafeClient → lets our notebook communicate with the Jev service.
- Choice → lets us define a decision/classification question with possible choices such as negative, neutral, and positive.

In [12]:
from openai import OpenAI

client = OpenAI(
    api_key=api_key,
    base_url="https://api.ai.kodekloud.com/v1"
)

print("KodeKloud AI client initialized successfully.")

KodeKloud AI client initialized successfully.


we are not actually connecting to OpenAI's API.

We're using the OpenAI Python SDK because KodeKloud provides an OpenAI-compatible API interface.

In [13]:
test_text = "I absolutely loved this movie. It was amazing!"

response = client.chat.completions.create(
    model="typesafe/jev-1.13.0",
    messages=[
        {
            "role": "user",
            "content": test_text
        }
    ],
    response_format={
        "type": "questions",
        "questions": {
            "sentiment": {
                "type": "choice",
                "instructions": "What is the sentiment of the provided text?",
                "criteria": {
                    "negative": "The text expresses negative sentiment.",
                    "neutral": "The text expresses neutral sentiment.",
                    "positive": "The text expresses positive sentiment."
                }
            }
        }
    }
)

print(response)

ChatCompletion(id='rqsty-cmpl-4a63f770-0000-4c61-991c-b7322c7507ba', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{"sentiment":{"choice":"positive","confidence":1,"probabilities":{"negative":0,"neutral":0,"positive":1},"type":"choice"}}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, provider_specific_fields={'refusal': None}), provider_specific_fields={})], created=1792603443, model='typesafe/jev-1.13.0', object='chat.completion', moderation=None, service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=39, prompt_tokens=346, total_tokens=385, completion_tokens_details=None, prompt_tokens_details=None, cost=1.4532e-05))


In [14]:
import json

jev_result = json.loads(response.choices[0].message.content)

print("Prediction:", jev_result["sentiment"]["choice"])
print("Confidence:", jev_result["sentiment"]["confidence"])
print("Probabilities:", jev_result["sentiment"]["probabilities"])

Prediction: positive
Confidence: 1
Probabilities: {'negative': 0, 'neutral': 0, 'positive': 1}


In [15]:
def predict_with_jev(text):
    response = client.chat.completions.create(
        model="typesafe/jev-1.13.0",
        messages=[
            {
                "role": "user",
                "content": text
            }
        ],
        response_format={
            "type": "questions",
            "questions": {
                "sentiment": {
                    "type": "choice",
                    "instructions": "What is the sentiment of the provided text?",
                    "criteria": {
                        "negative": "The text expresses negative sentiment.",
                        "neutral": "The text expresses neutral sentiment.",
                        "positive": "The text expresses positive sentiment."
                    }
                }
            }
        }
    )

    result = json.loads(response.choices[0].message.content)

    return {
        "prediction": result["sentiment"]["choice"],
        "confidence": result["sentiment"]["confidence"],
        "probabilities": result["sentiment"]["probabilities"]
    }

In [16]:
test_texts = [
    "I absolutely loved this movie. It was amazing!",
    "The movie was okay. Nothing particularly special.",
    "I hated this movie. It was terrible."
]

for text in test_texts:
    result = predict_with_jev(text)

    print("Text:", text)
    print("Prediction:", result["prediction"])
    print("Confidence:", result["confidence"])
    print("Probabilities:", result["probabilities"])
    print("-" * 60)

Text: I absolutely loved this movie. It was amazing!
Prediction: positive
Confidence: 1
Probabilities: {'negative': 0, 'neutral': 0, 'positive': 1}
------------------------------------------------------------
Text: The movie was okay. Nothing particularly special.
Prediction: neutral
Confidence: 0.94
Probabilities: {'negative': 0.04, 'neutral': 0.96, 'positive': 0}
------------------------------------------------------------
Text: I hated this movie. It was terrible.
Prediction: negative
Confidence: 1
Probabilities: {'negative': 1, 'neutral': 0, 'positive': 0}
------------------------------------------------------------


In [17]:
import pandas as pd
import numpy as np
df = pd.read_csv('test.csv', encoding='latin1')

df = df[['text', 'sentiment']]

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (14402, 2)

Column names:
['text', 'sentiment']

Missing values:
text         0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
negative    9079
neutral     3046
positive    2277
Name: count, dtype: int64

First 5 rows:


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [18]:
df = df.dropna(subset=['text', 'sentiment']).copy()

df.reset_index(drop=True, inplace=True)

print("Dataset shape after removing missing values:", df.shape)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

Dataset shape after removing missing values: (14402, 2)

Missing values after cleaning:
text         0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
negative    9079
neutral     3046
positive    2277
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split
X = df['text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.125,
    random_state=42,
    stratify=y_train
)

print("Training samples:", len(X_train_final))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

print("\nTraining distribution:")
print(y_train_final.value_counts())

print("\nValidation distribution:")
print(y_val.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

Training samples: 10080
Validation samples: 1441
Test samples: 2881

Training distribution:
sentiment
negative    6355
neutral     2132
positive    1593
Name: count, dtype: int64

Validation distribution:
sentiment
negative    908
neutral     305
positive    228
Name: count, dtype: int64

Test distribution:
sentiment
negative    1816
neutral      609
positive     456
Name: count, dtype: int64


In [20]:
from sklearn.model_selection import train_test_split

# Remove rows with missing text or sentiment
df = df.dropna(subset=["text", "sentiment"]).copy()

# Reset index
df.reset_index(drop=True, inplace=True)

# Separate text and labels
X = df["text"]
y = df["sentiment"]

# Recreate the original 20% test split
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Put the test data back into a DataFrame
test_df = pd.DataFrame({
    "text": X_test,
    "sentiment": y_test
})

print("Test samples:", len(test_df))
print("\nTest distribution:")
print(test_df["sentiment"].value_counts())

print("\nFirst 5 test examples:")
display(test_df.head())

Test samples: 2881

Test distribution:
sentiment
negative    1816
neutral      609
positive     456
Name: count, dtype: int64

First 5 test examples:


,text,sentiment
12928,"@AmericanAir got it, so it's just paying extra...",negative
3363,@united Need to track lost luggage being shipp...,negative
10604,@USAirways Can you help me update KTN on the p...,negative
10078,"@USAirways how aggravating, zone 5 boarded, ov...",negative
10334,@USAirways on hold 3 hours !!!!!! What's the n...,negative


In [21]:
sample_test = test_df.head(10).copy()

jev_predictions = []

for text in sample_test["text"]:
    result = predict_with_jev(text)
    jev_predictions.append(result)

sample_test["jev_prediction"] = [
    result["prediction"] for result in jev_predictions
]

sample_test["jev_confidence"] = [
    result["confidence"] for result in jev_predictions
]

print("Jev predictions for first 10 test examples:")
display(sample_test[[
    "text",
    "sentiment",
    "jev_prediction",
    "jev_confidence"
]])

Jev predictions for first 10 test examples:


,text,sentiment,jev_prediction,jev_confidence
12928,"@AmericanAir got it, so it's just paying extra...",negative,negative,1.00
3363,@united Need to track lost luggage being shipp...,negative,negative,0.77
10604,@USAirways Can you help me update KTN on the p...,negative,negative,1.00
10078,"@USAirways how aggravating, zone 5 boarded, ov...",negative,negative,1.00
10334,@USAirways on hold 3 hours !!!!!! What's the n...,negative,negative,1.00
6705,"@SouthwestAir Guys, we've got to do something ...",neutral,negative,1.00
2912,@united DM sent.,neutral,neutral,0.97
1235,@united love the new 1st class breakfast!,positive,positive,1.00
4951,@SouthwestAir thanks for the response - flight...,negative,negative,0.20
2296,"@united 2/2 Instead, had to wait on plane from...",negative,negative,1.00


In [22]:
print("Test set shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nClass distribution:")
print(test_df["sentiment"].value_counts())

print("\nFirst 5 examples:")
print(test_df[["text", "sentiment"]].head())

Test set shape: (2881, 2)
Columns: ['text', 'sentiment']

Class distribution:
sentiment
negative    1816
neutral      609
positive     456
Name: count, dtype: int64

First 5 examples:
                                                    text sentiment
12928  @AmericanAir got it, so it's just paying extra...  negative
3363   @united Need to track lost luggage being shipp...  negative
10604  @USAirways Can you help me update KTN on the p...  negative
10078  @USAirways how aggravating, zone 5 boarded, ov...  negative
10334  @USAirways on hold 3 hours !!!!!! What's the n...  negative


In [23]:
# Test Jev on a small batch before evaluating the full test set
sample_test = test_df.sample(n=20, random_state=42).copy()

jev_predictions = []

for text in sample_test["text"]:
    result = predict_with_jev(text)
    jev_predictions.append(result)

sample_test["jev_prediction"] = [
    result["prediction"] for result in jev_predictions
]

sample_test["jev_confidence"] = [
    result["confidence"] for result in jev_predictions
]

print("Sample size:", len(sample_test))
display(
    sample_test[
        ["text", "sentiment", "jev_prediction", "jev_confidence"]
    ]
)

Sample size: 20


,text,sentiment,jev_prediction,jev_confidence
91,"@VirginAmerica hi, i did not get points on my ...",neutral,negative,0.33
619,@united What's going on with your website? I'm...,negative,negative,1.00
5347,@SouthwestAir customer svc worse than the #Win...,negative,negative,1.00
9945,@USAirways you Cancelled Flighted &amp; resch...,negative,negative,1.00
10888,@USAirways doesn't seem likely bc your team fa...,negative,negative,1.00
1812,"@united welcome to our world, that's a snow wo...",neutral,neutral,0.55
13817,@AmericanAir VERY upset that I cannot select s...,negative,negative,1.00
13231,@AmericanAir Your customer service sucks! You ...,negative,negative,1.00
6727,@JetBlue Ok and in Dominican Republic what num...,neutral,neutral,1.00
4172,@united 3359 which you finally Cancelled Fligh...,negative,negative,0.79


In [24]:
# Check prediction labels and confidence values
valid_labels = {"negative", "neutral", "positive"}

print("All predictions have valid labels:",
      all(p["prediction"] in valid_labels for p in jev_predictions))

print("All confidence values are between 0 and 1:",
      all(0 <= p["confidence"] <= 1 for p in jev_predictions))

print("Number of predictions:", len(jev_predictions))

All predictions have valid labels: True
All confidence values are between 0 and 1: True
Number of predictions: 20


In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_true = sample_test["sentiment"]
y_pred = sample_test["jev_prediction"]

print(f"Sample Accuracy: {accuracy_score(y_true, y_pred):.2%}")

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["negative", "neutral", "positive"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=["negative", "neutral", "positive"]
    )
)

Sample Accuracy: 85.00%

Classification Report:
              precision    recall  f1-score   support

    negative       0.80      1.00      0.89        12
     neutral       1.00      0.60      0.75         5
    positive       1.00      0.67      0.80         3

    accuracy                           0.85        20
   macro avg       0.93      0.76      0.81        20
weighted avg       0.88      0.85      0.84        20


Confusion Matrix:
[[12  0  0]
 [ 2  3  0]
 [ 1  0  2]]


In [26]:
# CELL 18 — Google Drive setup
# Run this cell first. It creates the permanent folder and checkpoint paths.

from google.colab import drive
import os
import json
import shutil #Provides file operations such as copying files.

drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/Sentiment_Analysis/Jev_Evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "jev_predictions_checkpoint.json")
BLOCKED_FILE = os.path.join(OUTPUT_DIR, "jev_blocked_indices.json")

# Copy older files from temporary Colab storage only if Drive has no copy yet.
for filename in ["jev_predictions_checkpoint.json", "jev_blocked_indices.json"]:
    local_path = f"/content/{filename}"
    drive_path = os.path.join(OUTPUT_DIR, filename)
    if os.path.exists(local_path) and not os.path.exists(drive_path):
        shutil.copy2(local_path, drive_path)
        print(f"Copied existing local file to Drive: {filename}")

print("Google Drive is ready.")
print("Output folder:", OUTPUT_DIR)


Mounted at /content/drive
Google Drive is ready.
Output folder: /content/drive/MyDrive/Sentiment_Analysis/Jev_Evaluation


In [27]:
# CELL 19 — Run Jev evaluation and save progress to Google Drive
# Run Cell 18 first. Requires test_df and predict_with_jev() from earlier cells.

import json
import os

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        prediction_results = json.load(f)
else:
    prediction_results = []

if os.path.exists(BLOCKED_FILE):
    with open(BLOCKED_FILE, "r", encoding="utf-8") as f:
        blocked_indices = set(json.load(f))
else:
    blocked_indices = set()

completed_indices = {int(item["index"]) for item in prediction_results}

print("Previously completed predictions:", len(completed_indices))
print("Previously blocked examples:", len(blocked_indices))
print("Total test examples:", len(test_df))

for idx in range(len(test_df)):
    if idx in completed_indices or idx in blocked_indices:
        continue

    row = test_df.iloc[idx]
    try:
        result = predict_with_jev(row["text"])
        prediction_results.append({
            "index": int(idx),
            "text": str(row["text"]),
            "sentiment": str(row["sentiment"]),
            "jev_prediction": str(result["prediction"]),
            "jev_confidence": float(result["confidence"])
        })
        completed_indices.add(idx)

    except Exception as e:
        error_message = str(e)
        print(f"Error at example {idx}: {error_message}")
        if "Content blocked by Model Armor" in error_message:
            blocked_indices.add(idx)
            print(f"Skipping blocked example {idx} and continuing.")
        else:
            # Save progress before stopping on an unexpected API error.
            with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
                json.dump(prediction_results, f, ensure_ascii=False, indent=2)
            with open(BLOCKED_FILE, "w", encoding="utf-8") as f:
                json.dump(sorted(blocked_indices), f, indent=2)
            print("Unexpected API error. Progress was saved; stopping.")
            break

    # Save progress after every processed example.
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(prediction_results, f, ensure_ascii=False, indent=2)
    with open(BLOCKED_FILE, "w", encoding="utf-8") as f:
        json.dump(sorted(blocked_indices), f, indent=2)

    processed = len(completed_indices) + len(blocked_indices)
    if processed % 50 == 0:
        print(f"Processed {processed}/{len(test_df)} | Predictions: {len(completed_indices)} | Blocked: {len(blocked_indices)}")

print("\nEvaluation progress saved to Google Drive.")
print("Successful predictions:", len(completed_indices))
print("Blocked examples:", len(blocked_indices))
print("Remaining examples:", len(test_df) - len(completed_indices) - len(blocked_indices))
print("Checkpoint:", CHECKPOINT_FILE)
print("Blocked records:", BLOCKED_FILE)


Previously completed predictions: 2877
Previously blocked examples: 4
Total test examples: 2881

Evaluation progress saved to Google Drive.
Successful predictions: 2877
Blocked examples: 4
Remaining examples: 0
Checkpoint: /content/drive/MyDrive/Sentiment_Analysis/Jev_Evaluation/jev_predictions_checkpoint.json
Blocked records: /content/drive/MyDrive/Sentiment_Analysis/Jev_Evaluation/jev_blocked_indices.json


In [29]:

# CELL 20 — Evaluate predictions and display metrics only
# Reads saved predictions from Cell 19.
# Does not save any final output files or call the Jev API.

import os
import json
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Check that the checkpoint exists
if not os.path.exists(CHECKPOINT_FILE):
    raise FileNotFoundError("Checkpoint file not found. Run Cell 19 first.")

# 2. Load previously saved predictions
with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
    prediction_results = json.load(f)

if not prediction_results:
    raise ValueError("No successful predictions are saved yet. Run Cell 19 first.")

# 3. Organize predictions and remove duplicate test indices
results_df = (
    pd.DataFrame(prediction_results)
    .drop_duplicates(subset="index", keep="last")
    .sort_values("index")
    .reset_index(drop=True)
)

# 4. Load blocked indices, if available
if os.path.exists(BLOCKED_FILE):
    with open(BLOCKED_FILE, "r", encoding="utf-8") as f:
        blocked_indices = set(json.load(f))
else:
    blocked_indices = set()

# 5. Display evaluation progress
total_samples = len(test_df)
successful = len(results_df)
blocked = len(blocked_indices)
remaining = max(0, total_samples - successful - blocked)
coverage = successful / total_samples if total_samples else 0.0

print("=" * 60)
print("JEV SENTIMENT ANALYSIS — EVALUATION RESULTS")
print("=" * 60)

print(f"Total test samples:       {total_samples}")
print(f"Successful predictions:   {successful}")
print(f"Blocked examples:         {blocked}")
print(f"Remaining examples:       {remaining}")
print(f"Evaluation coverage:      {coverage:.2%}")

# 6. Compare actual labels with Jev predictions
labels = ["negative", "neutral", "positive"]
y_true = results_df["sentiment"]
y_pred = results_df["jev_prediction"]

accuracy = accuracy_score(y_true, y_pred)

print(f"\nJev Test Accuracy: {accuracy:.2%}")

# 7. Display precision, recall, F1-score, and support
print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0
    )
)

# 8. Display the confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=labels)

print("\nConfusion Matrix (rows=actual, columns=predicted):")
print("Label order:", labels)
print(cm)

# 9. Display a readable confusion matrix table
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in labels],
    columns=[f"Predicted {label}" for label in labels]
)

display(cm_df)

print("\nEvaluation complete. ")


JEV SENTIMENT ANALYSIS — EVALUATION RESULTS
Total test samples:       2881
Successful predictions:   2877
Blocked examples:         4
Remaining examples:       0
Evaluation coverage:      99.86%

Jev Test Accuracy: 84.01%

Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.94      0.92      1815
     neutral       0.79      0.51      0.62       607
    positive       0.70      0.87      0.78       455

    accuracy                           0.84      2877
   macro avg       0.79      0.77      0.77      2877
weighted avg       0.84      0.84      0.83      2877


Confusion Matrix (rows=actual, columns=predicted):
Label order: ['negative', 'neutral', 'positive']
[[1712   68   35]
 [ 166  308  133]
 [  42   16  397]]


,Predicted negative,Predicted neutral,Predicted positive
Actual negative,1712,68,35
Actual neutral,166,308,133
Actual positive,42,16,397



Evaluation complete. 


## Final Conclusion — Jev Sentiment Analysis

In this notebook, I evaluated the Jev model (`typesafe/jev-1.13.0`) for three-class sentiment classification using an OpenAI-compatible API provided by KodeKloud. The dataset contained 14,402 text samples labeled as negative, neutral, or positive, and the model was evaluated on a test set of 2,881 samples.

The evaluation successfully obtained predictions for 2,877 samples, achieving **84.01% accuracy**, a **0.77 macro F1-score**, and a **0.83 weighted F1-score**. The model performed best on negative sentiment, with an F1-score of 0.92, followed by positive sentiment at 0.78. Neutral sentiment was more challenging, with an F1-score of 0.62, indicating difficulty distinguishing neutral text from other sentiment classes.

Four test examples were blocked by the API's content-safety mechanism and were excluded from the classification metrics. Therefore, the reported metrics are based on the 2,877 successfully evaluated samples, representing 99.86% evaluation coverage.

Overall, this experiment demonstrates that an API-based language model can perform sentiment classification without training a separate model on the dataset. However, neutral sentiment remains an area for improvement. These results can be compared with the locally trained Logistic Regression, PyTorch neural network, and DistilBERT models to assess their relative performance, cost, and practical trade-offs.